In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_pickle('../results/HelmLiteRejectors/results_compressed.pkl')
df["epoch"] = df["epoch"].astype(int)

In [ ]:
sub = df

runs = sub.run.unique()
k_vals, std_of_means = [], []
for k in range(1, len(runs)+1):
    means = []
    for _ in range(100):                              
        chosen = np.random.choice(runs, size=k, replace=False)
        m = sub[sub.run.isin(chosen)].groupby("epoch").auroc.mean().mean()
        means.append(m)
    k_vals.append(k)
    std_of_means.append(np.std(means))

plt.figure()
plt.plot(k_vals, std_of_means, marker="o")
plt.xlabel("Number of runs sampled")
plt.ylabel("Std of mean AUROC")
plt.title("Run-to-run variability vs. # runs")
plt.grid(True)
plt.show()

In [ ]:
agg = (sub.groupby("epoch")["auroc"].agg(mean="mean", std="std").sort_index())

x = agg.index.to_numpy()
m = agg["mean"].to_numpy()
s = agg["std"].to_numpy()

plt.figure()
plt.plot(x, m, "-o")
plt.fill_between(x, m - s, m + s, alpha=0.2)
plt.xlabel("Epoch")
plt.ylabel("AUROC")
plt.title("Mean ± std AUROC by epoch")
plt.grid(True)
plt.show()

best_epoch = agg["mean"].idxmax()
print("Best mean epoch:", best_epoch)

In [ ]:
best_per_hp = ( #picks best epoch
    df
    .groupby(["model","f1","bleu","run","epoch"])
    .auroc.mean()
    .reset_index()
    .groupby(["model","f1","bleu","run"])
    .apply(lambda d: d.loc[d.auroc.idxmax()])
    .reset_index(drop=True)
)

hp_stats = (
    best_per_hp
    .groupby(["f1", "bleu"])            
    .auroc
    .mean()
    .reset_index(name="mean")           
)

pivot = hp_stats.pivot(index="f1", columns="bleu", values="mean")

plt.figure(figsize=(8,6), dpi=150)
ax = sns.heatmap(
    pivot,
    annot=True,      
    fmt=".3f",      
    cmap=sns.color_palette("magma", as_cmap=True),
    cbar_kws={"label": "Mean AUROC"},
    annot_kws={"size": 16}
)

cbar = ax.collections[0].colorbar
cbar.set_label("Mean AUROC", fontsize=16)
cbar.ax.tick_params(labelsize=14)

plt.xlabel("BLEU-4", fontsize=16)
plt.ylabel("F1", fontsize=16)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.tight_layout()
plt.savefig("../results/figures/Barrier1_threshold-heatmap.pdf", dpi=300, pad_inches=0.0, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure()
sns.boxplot(data=best_per_hp,x="f1", y="auroc", hue="bleu")
plt.title("AUROC distribution by hyperparams")
plt.show()

In [ ]:
best_per_hp = ( #picks best epoch
    df
    .groupby(["model", "similarity", "run", "epoch"])  
    .auroc.mean()                                                   
    .reset_index()
    .groupby(["model", "similarity", "run"])
    .apply(lambda d: d.loc[d.auroc.idxmax()])
    .reset_index(drop=True)
)

sim_stats = (
    best_per_hp
    .groupby("similarity")
    .auroc
    .mean()
    .reset_index(name="mean_auroc")
)

sim_stats = sim_stats.sort_values(by="mean_auroc", ascending=False)

plt.figure(figsize=(6, 4))
plt.bar(sim_stats["similarity"], sim_stats["mean_auroc"])
plt.xlabel("Similarity")
plt.ylabel("Mean AUROC")
plt.title("Mean AUROC by Similarity")
plt.tight_layout()
plt.show()

In [ ]:
sim_stats